# Delta Chat Bots REST API Connector - Google Colab

Este notebook permite ejecutar el Delta Chat Bots REST API Connector en Google Colab.

## Instrucciones:
1. Ejecuta todas las celdas en orden
2. Configura tus variables de entorno en la celda correspondiente
3. Inicia el bot y mantén el notebook ejecutándose

**Nota importante:** Google Colab tiene limitaciones para servidores HTTP públicos. Para exponer la API, usaremos ngrok o cloudflared.

In [ ]:
# Instalar dependencias del sistema requeridas por deltabot-cli
!apt-get update
!apt-get install -y python3-dev libffi-dev build-essential

In [ ]:
# Clonar el repositorio (si no estás ejecutando desde un repo clonado)
# Descomenta estas líneas si necesitas clonar el repositorio
# !git clone https://github.com/tu-usuario/Delta-Chat-Bots---REST-API.git
# %cd Delta-Chat-Bots---REST-API

In [ ]:
# Instalar dependencias de Python
!pip install -r requirements.txt -q

In [ ]:
# Instalar pyngrok para exponer el servidor local
!pip install pyngrok -q

In [ ]:
# Configurar variables de entorno
# IMPORTANTE: Reemplaza estos valores con tus credenciales reales

import os

# Configuración del bot
os.environ['BOT_CLI_NAME'] = 'ColabBot'
os.environ['LOG_LEVEL'] = 'info'

# Webhook (opcional)
os.environ['ENABLE_WEBHOOK'] = 'false'
# os.environ['WEBHOOK_URL'] = 'https://tu-webhook.com/endpoint'
# os.environ['WEBHOOK_AUTH_TOKEN'] = 'tu-token-secreto'

# API Configuration - En Colab, usamos 0.0.0.0 para permitir acceso externo
os.environ['API_HOST'] = '0.0.0.0'
os.environ['API_PORT'] = '8000'

# ¡IMPORTANTE! Cambia esto por una clave segura
os.environ['API_KEY'] = 'cambia-esta-clave-por-una-segura'

# Directorio de medios
os.environ['MEDIA_DIR'] = '/content/DeltaChatBotsRestAPI_media'

print("Variables de entorno configuradas")
print(f"API Key: {os.environ.get('API_KEY')}")
print(f"API Host: {os.environ.get('API_HOST')}")
print(f"API Port: {os.environ.get('API_PORT')}")

In [ ]:
# Configurar ngrok para exponer el servidor
# Necesitas un token de ngrok gratuito de https://ngrok.com/signup

from pyngrok import ngrok

# Reemplaza con tu token de ngrok
NGROK_AUTH_TOKEN = "tu-ngrok-token-aqui"

if NGROK_AUTH_TOKEN and NGROK_AUTH_TOKEN != "tu-ngrok-token-aqui":
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    
    # Crear túnel HTTP
    public_url = ngrok.connect(8000)
    print(f"\n" + "="*50)
    print(f"Delta Chat REST API expuesta en:")
    print(f"{public_url}")
    print(f"="*50 + "\n")
    print(f"URL pública: {public_url}")
    print(f"No olvides usar tu API_KEY en las cabeceras de las requests")
else:
    print("⚠️ ADVERTENCIA: No has configurado tu token de ngrok")
    print("Obtén uno gratis en: https://ngrok.com/signup")
    print("El servidor se ejecutará pero no será accesible públicamente")

In [ ]:
# Crear cuenta del bot (SOLO LA PRIMERA VEZ)
# Ejecuta esta celda una vez para inicializar el bot
# Luego comenta o salta esta celda en ejecuciones futuras

# !python main.py init DCACCOUNT:https://nine.testrun.org/new
# !python main.py config displayname "Mi Bot REST API en Colab"

print("Para crear una cuenta de bot, ejecuta en la terminal:")
print("  !python main.py init DCACCOUNT:https://nine.testrun.org/new")
print("  !python main.py config displayname 'Mi Bot REST API'")
print("\nLuego genera un enlace de invitación con:")
print("  !python main.py link")

In [ ]:
# Verificar cuentas existentes
!python main.py list

In [ ]:
# Iniciar el bot y la API REST
# Esta celda se ejecutará indefinidamente hasta que detengas el notebook

import subprocess
import sys

print("Iniciando Delta Chat Bot con REST API...")
print("Presiona Ctrl+C o el botón de detener para terminar el bot")
print("\n" + "="*50)

# Ejecutar el bot
!python main.py serve

## Uso de la API

Una vez que el bot esté corriendo, puedes usar la API desde cualquier cliente HTTP:

### Health Check:
```bash
curl -X GET https://tu-url-ngrok.ngrok.io/health \
  -H "Authorization: Bearer tu-api-key"
```

### Enviar mensaje:
```bash
curl -X POST https://tu-url-ngrok.ngrok.io/rpc \
  -H "Authorization: Bearer tu-api-key" \
  -H "Content-Type: application/json" \
  -d '{
    "method": "send_msg",
    "params": [1, 10, {"text": "Hola desde Colab"}]
  }'
```

### Obtener información de la cuenta:
```bash
curl -X POST https://tu-url-ngrok.ngrok.io/rpc \
  -H "Authorization: Bearer tu-api-key" \
  -H "Content-Type: application/json" \
  -d '{
    "method": "get_account_info",
    "params": [1]
  }'
```

## Notas Importantes

1. **Persistencia**: Los datos del bot se guardan en `/root/.local/share/deltabot-cli/`. En Colab, estos datos pueden perderse cuando la sesión termina.

2. **Ngrok**: La versión gratuita de ngrok tiene límites. Considera actualizar a un plan pago o usar alternativas como cloudflared.

3. **Timeout**: Colab desconecta sesiones inactivas después de ~90 minutos. Mantén el notebook activo.

4. **Seguridad**: Nunca compartas tu API_KEY públicamente. Usa variables de entorno o Google Secrets.

5. **Para producción**: Considera usar Render, Railway, o un VPS en lugar de Colab.